### Turingmaschinen

[Videos](https://www.youtube.com/playlist?list=PLWeMgMhRDsIFksEYpzuntcAtE6dmphQas)

Dieses Notebook setzt die Notebooks **Reguläre Sprachen** und **Kellerautomaten** voraus.

Ein endlicher Automat hat nur seinen Zustand als Gedächtnis, ein Kellerautomat zusätzlich einen Keller. Die Sprache
$\{a^nb^nc^n\}$ kann aber auch ein Kellerautomat nicht erkennen. Alan Turing hat 1936 ein Maschinenmodell beschrieben,
das alles kann, was überhaupt ein Computer kann: die **Turingmaschine**.

#### Aufbau

Eine **Turingmaschine (TM)** besteht aus

- einem unendlich langen **Band**, eingeteilt in Felder. In jedem Feld steht ein Zeichen. Leere Felder enthalten das
  Leerzeichen $\sqcup$ (Blank).
- einem **Lese-/Schreibkopf**, der auf einem Feld steht und sich nach links und rechts bewegen kann.
- einer **endlichen Kontrolle**, die sich den aktuellen Zustand merkt.

<img src="bild1.png" width="400">

In jedem Schritt liest die TM das Zeichen unter dem Kopf. Abhängig vom Zustand und diesem Zeichen schreibt sie ein
Zeichen in das Feld, bewegt den Kopf um ein Feld nach links (L), nach rechts (R) oder gar nicht (N) und wechselt den
Zustand.

Der Unterschied zum Kellerautomaten: Die TM kann überall auf dem Band lesen und schreiben, beliebig oft hin und her
fahren und die Eingabe selbst überschreiben.

#### Definition

Eine **deterministische Turingmaschine** besteht aus

1. $Q$, einer endlichen Menge von Zuständen,
2. $\Sigma$, einem endlichen Eingabealphabet,
3. $\Gamma \supseteq \Sigma \cup \{\sqcup\}$, einem endlichen Bandalphabet, $\sqcup \notin \Sigma$,
4. $\delta$, einer Übergangsfunktion: $\delta(q, a) = (p, b, R)$ heißt *im Zustand $q$ mit $a$ unter dem Kopf: schreibe
   $b$, gehe nach rechts und wechsle in den Zustand $p$*,
5. $s \in Q$, einem Startzustand,
6. $F \subseteq Q$, einer Menge von Endzuständen.

Im Zustandsdiagramm steht am Pfeil von $q$ nach $p$ die Beschriftung $a \mid b, R$: gelesen $\mid$ geschrieben,
Richtung.

**Vereinbarungen:**

- Am Anfang steht die Eingabe auf dem Band, sonst nur Blanks. Die TM startet im Startzustand auf dem ersten Zeichen der
  Eingabe.
- Sie **hält und akzeptiert**, sobald sie in einen Endzustand kommt.
- Sie **hält und lehnt ab**, wenn es für den Zustand und das gelesene Zeichen keinen Übergang gibt.
- Es kann auch passieren, dass sie **nie hält**.

Die TM **akzeptiert** die Sprache $L$, wenn sie genau die Wörter aus $L$ akzeptiert. Bei Wörtern, die nicht zu $L$
gehören, hält sie in einem Zustand, der kein Endzustand ist, oder sie hält gar nicht.

### Beispiel 1

<img src="tm_beispiel1.png" width="560">

Die Abarbeitung beschreibt man mit **Konfigurationen**: Zustand und Bandinhalt, das Zeichen unter dem Kopf ist
markiert. Für die Eingabe $100$:

| Zustand | Band |
|---|---|
| $q_0$ | <u>1</u>00 |
| $q_1$ | 0<u>0</u>0 |
| $q_1$ | 00<u>0</u> |
| $q_1$ | 000<u>⊔</u> |
| $q_2$ | 00<u>0</u>1 |
| $q_2$ | 0<u>0</u>01 |
| $q_2$ | <u>0</u>001 |
| $q_2$ | <u>⊔</u>0001 |
| $q_3$ | <u>0</u>001 |

Die TM ersetzt die erste $1$ durch eine $0$, fährt nach rechts bis zum Ende, hängt eine $1$ an und fährt zurück an
den Anfang. Wörter, die mit $0$ beginnen, lehnt sie sofort ab: Für $q_0$ und $0$ gibt es keinen Übergang.

In JFLAP schreibt man die Übergänge als `1 ; 0 , R`, das Blank als leeres Kästchen und $S$ (stay) statt $N$.

### Turingmaschinen in Python

Die Übergangsfunktion speichern wir als Dictionary: `(zustand, zeichen)` wird auf `(neuer_zustand, schreiben,
richtung)` abgebildet. Das Blank schreiben wir als `_`. Das Band ist ein Dictionary von Position auf Zeichen. So kann
der Kopf beliebig weit nach links und rechts fahren.

In [1]:
beispiel1 = {
    'start': 'q0',
    'ende': {'q3'},
    'delta': {
        ('q0', '1'): ('q1', '0', 'R'),
        ('q1', '0'): ('q1', '0', 'R'),
        ('q1', '1'): ('q1', '1', 'R'),
        ('q1', '_'): ('q2', '1', 'L'),
        ('q2', '0'): ('q2', '0', 'L'),
        ('q2', '1'): ('q2', '1', 'L'),
        ('q2', '_'): ('q3', '_', 'R'),
    },
}

def bandtext(band, pos):
    '''Bandinhalt, das Zeichen unter dem Kopf in eckigen Klammern'''
    links = min(list(band) + [pos])
    rechts = max(list(band) + [pos])
    text = ''
    for i in range(links, rechts + 1):
        z = band.get(i, '_')
        text += f'[{z}]' if i == pos else z
    return text.strip('_')

def lauf(tm, wort, max_schritte=1000, zeigen=True):
    band = dict(enumerate(wort))
    zustand, pos = tm['start'], 0
    for schritt in range(max_schritte):
        if zeigen:
            print(f'{zustand:4}{bandtext(band, pos)}')
        if zustand in tm['ende']:
            return 'akzeptiert'
        zeichen = band.get(pos, '_')
        if (zustand, zeichen) not in tm['delta']:
            return 'abgelehnt'
        zustand, neu, richtung = tm['delta'][(zustand, zeichen)]
        band[pos] = neu
        if richtung == 'L':
            pos = pos - 1
        elif richtung == 'R':
            pos = pos + 1
    return f'kein Halt nach {max_schritte} Schritten'

lauf(beispiel1, '100')

q0  [1]00
q1  0[0]0
q1  00[0]
q1  000[_]
q2  00[0]1
q2  0[0]01
q2  [0]001
q2  [_]0001
q3  [0]001


'akzeptiert'

In [2]:
lauf(beispiel1, '0110')

q0  [0]110


'abgelehnt'

### Beispiel 2: Eine TM, die nicht immer hält

<img src="tm_beispiel2.png" width="380">

Welche Wörter über $\{0, 1\}$ akzeptiert diese TM? Gibt es Eingaben, für die sie nie hält?

In [3]:
beispiel2 = {
    'start': 's',
    'ende': {'q2'},
    'delta': {
        ('s', '_'): ('s', '_', 'R'),
        ('s', '0'): ('q1', '0', 'R'),
        ('s', '1'): ('q2', '1', 'N'),
        ('q1', '0'): ('q1', '0', 'R'),
        ('q1', '1'): ('q1', '1', 'R'),
        ('q1', '_'): ('q1', '_', 'R'),
    },
}
for w in ['1', '10', '0', '01', '']:
    print(f'{w:3}', lauf(beispiel2, w, zeigen=False))

1   akzeptiert
10  akzeptiert
0   kein Halt nach 1000 Schritten
01  kein Halt nach 1000 Schritten
    kein Halt nach 1000 Schritten


Die TM akzeptiert alle Wörter, die mit $1$ beginnen. Beginnt das Wort mit $0$ oder ist es leer, dann fährt sie für
immer nach rechts. Unser Programm bricht nach 1000 Schritten ab, die Maschine selbst würde nie halten.

Man kann einer TM von außen nicht immer ansehen, ob sie für eine Eingabe hält. Dass es dafür kein allgemeines Verfahren
gibt, zeigt das **Halteproblem** (siehe Notebook **Halteproblem**).

### Mehr als ein Kellerautomat: $a^nb^nc^n$

Kein Kellerautomat erkennt $\{a^nb^nc^n \mid n \ge 1\}$. Eine TM schafft es, indem sie die Zeichen abhakt: Sie
ersetzt das erste $a$ durch $X$, das erste $b$ durch $Y$, das erste $c$ durch $Z$, fährt zurück zum letzten $X$ und
beginnt von vorne. Sind alle $a$ abgehakt, prüft sie, ob nur noch $Y$ und $Z$ folgen.

<img src="tm_anbncn.png" width="700">

- $q_0$: nächstes $a$ abhaken, oder nach $q_4$, wenn kein $a$ mehr da ist
- $q_1$: nach rechts zum nächsten $b$, $q_2$: nach rechts zum nächsten $c$
- $q_3$: zurück nach links bis zum $X$
- $q_4$: prüfen, ob bis zum Ende nur noch $Y$ und $Z$ kommen

In [4]:
anbncn = {
    'start': 'q0',
    'ende': {'q5'},
    'delta': {
        ('q0', 'a'): ('q1', 'X', 'R'), ('q0', 'Y'): ('q4', 'Y', 'R'),
        ('q1', 'a'): ('q1', 'a', 'R'), ('q1', 'Y'): ('q1', 'Y', 'R'), ('q1', 'b'): ('q2', 'Y', 'R'),
        ('q2', 'b'): ('q2', 'b', 'R'), ('q2', 'Z'): ('q2', 'Z', 'R'), ('q2', 'c'): ('q3', 'Z', 'L'),
        ('q3', 'a'): ('q3', 'a', 'L'), ('q3', 'b'): ('q3', 'b', 'L'), ('q3', 'Y'): ('q3', 'Y', 'L'),
        ('q3', 'Z'): ('q3', 'Z', 'L'), ('q3', 'X'): ('q0', 'X', 'R'),
        ('q4', 'Y'): ('q4', 'Y', 'R'), ('q4', 'Z'): ('q4', 'Z', 'R'), ('q4', '_'): ('q5', '_', 'N'),
    },
}
lauf(anbncn, 'aabbcc')

q0  [a]abbcc
q1  X[a]bbcc
q1  Xa[b]bcc
q2  XaY[b]cc
q2  XaYb[c]c
q3  XaY[b]Zc
q3  Xa[Y]bZc
q3  X[a]YbZc
q3  [X]aYbZc
q0  X[a]YbZc
q1  XX[Y]bZc
q1  XXY[b]Zc
q2  XXYY[Z]c
q2  XXYYZ[c]
q3  XXYY[Z]Z
q3  XXY[Y]ZZ
q3  XX[Y]YZZ
q3  X[X]YYZZ
q0  XX[Y]YZZ
q4  XXY[Y]ZZ
q4  XXYY[Z]Z
q4  XXYYZ[Z]
q4  XXYYZZ[_]
q5  XXYYZZ[_]


'akzeptiert'

In [5]:
import itertools
import re

def alle_woerter(alphabet, n):
    for laenge in range(n + 1):
        for t in itertools.product(alphabet, repeat=laenge):
            yield ''.join(t)

def in_L(w):
    m = re.fullmatch('(a*)(b*)(c*)', w)
    return m is not None and len(m[1]) == len(m[2]) == len(m[3]) > 0

fehler = [w for w in alle_woerter('abc', 9) if (lauf(anbncn, w, zeigen=False) == 'akzeptiert') != in_L(w)]
print(len(fehler), 'Fehler')

0 Fehler


### Turingmaschinen berechnen Funktionen

Eine TM kann auch etwas ausrechnen: Wenn sie hält, steht das Ergebnis auf dem Band. Diese TM addiert $1$ zu einer
Binärzahl:

<img src="tm_plus1.png" width="420">

Sie fährt nach rechts ans Ende der Zahl. Von dort geht sie nach links: Jede $1$ wird zu $0$ (Übertrag), die erste $0$
oder das Blank vor der Zahl wird zu $1$, dann ist sie fertig. Genauso addiert man schriftlich.

In [6]:
plus1 = {
    'start': 'q0',
    'ende': {'q2'},
    'delta': {
        ('q0', '0'): ('q0', '0', 'R'), ('q0', '1'): ('q0', '1', 'R'), ('q0', '_'): ('q1', '_', 'L'),
        ('q1', '1'): ('q1', '0', 'L'), ('q1', '0'): ('q2', '1', 'N'), ('q1', '_'): ('q2', '1', 'N'),
    },
}
lauf(plus1, '1011')
print()
lauf(plus1, '111')

q0  [1]011
q0  1[0]11
q0  10[1]1
q0  101[1]
q0  1011[_]
q1  101[1]
q1  10[1]0
q1  1[0]00
q2  1[1]00

q0  [1]11
q0  1[1]1
q0  11[1]
q0  111[_]
q1  11[1]
q1  1[1]0
q1  [1]00
q1  [_]000
q2  [1]000


'akzeptiert'

Eine Funktion heißt **(Turing-)berechenbar**, wenn es eine TM gibt, die für jede Eingabe hält und das Ergebnis auf das
Band schreibt. Mit einer TM kann man addieren, multiplizieren, vergleichen, sortieren, ... Die Programme werden schnell
lang, aber das Prinzip bleibt einfach.

#### Der fleißige Biber

Wie viele Schritte kann eine TM mit wenigen Zuständen höchstens machen, wenn sie auf leerem Band startet und
irgendwann hält? Das ist die Frage nach dem **fleißigen Biber** (busy beaver). Mit zwei Zuständen (und dem Haltezustand
$H$) ist der Rekord 6 Schritte:

In [7]:
biber = {
    'start': 'A',
    'ende': {'H'},
    'delta': {
        ('A', '_'): ('B', '1', 'R'), ('A', '1'): ('B', '1', 'L'),
        ('B', '_'): ('A', '1', 'L'), ('B', '1'): ('H', '1', 'R'),
    },
}
lauf(biber, '')

A   [_]
B   1[_]
A   [1]1
B   [_]11
A   [_]111
B   1[1]11
H   11[1]1


'akzeptiert'

Für 5 Zustände wurde erst 2024 bewiesen, dass der Rekord 47 176 870 Schritte beträgt. Für 6 Zustände ist er
unbekannt und unvorstellbar groß. Eine allgemeine Formel kann es nicht geben, die Biber-Funktion ist nicht berechenbar:
Sonst könnte man das Halteproblem lösen, indem man eine TM einfach so viele Schritte laufen lässt.

### Varianten

Man kann Turingmaschinen erweitern: mehrere Bänder, ein Band, das in zwei Dimensionen läuft, Nichtdeterminismus wie
beim NEA. Alle diese Varianten können **genau dasselbe**, sie sind nur manchmal schneller oder einfacher zu
programmieren. Auch ein Kellerautomat mit zwei Kellern ist so mächtig wie eine TM (siehe Notebook **Kellerautomaten**).

Die Turingmaschine steht damit an der Spitze der Chomsky-Hierarchie:

| Modell | Speicher | Sprachen |
|---|---|---|
| endlicher Automat | nur der Zustand | regulär (Typ 3) |
| Kellerautomat | ein Keller | kontextfrei (Typ 2) |
| linear beschränkter Automat | Band so lang wie die Eingabe | kontextsensitiv (Typ 1) |
| Turingmaschine | unbegrenztes Band | Typ 0 |

### Die Church-Turing-These

Turingmaschinen berechnen (partielle) Funktionen: Sie bekommen eine Eingabe und liefern, wenn sie halten, eine Ausgabe.

> **Church-Turing-These:** Die Turing-berechenbaren Funktionen sind genau die Funktionen, die im intuitiven Sinn
> überhaupt berechenbar sind.

Das ist eine These, kein Satz: *Im intuitiven Sinn berechenbar* ist kein mathematischer Begriff, man kann es nicht
beweisen. Aber alle Versuche, Berechenbarkeit genau zu fassen (Turingmaschinen, Churchs Lambda-Kalkül, Registermaschinen,
Programmiersprachen), haben dieselbe Klasse von Funktionen ergeben. Deshalb gilt die TM als formales Modell für
Algorithmen: Was keine TM ausführen kann, kann kein Algorithmus.

#### Turing-vollständig

Eine Programmiersprache heißt **Turing-vollständig**, wenn man in ihr jede Turingmaschine simulieren kann. Python ist
Turing-vollständig, die Funktion `lauf` oben zeigt es (bis auf den begrenzten Speicher eines echten Computers).
Umgekehrt kann eine TM jedes Python-Programm ausführen. Alle üblichen Programmiersprachen können also genau dasselbe
berechnen.

`lauf` ist ein Programm, das eine *beliebige* Turingmaschine ausführt, deren Beschreibung es als Eingabe bekommt. So ein
Programm gibt es auch als Turingmaschine: die **universelle Turingmaschine**. Sie ist das Vorbild für den Computer, der
Programme als Daten lädt und ausführt.

#### Entscheiden und Erkennen

Eine TM **entscheidet** eine Sprache, wenn sie bei jeder Eingabe hält und genau die Wörter der Sprache akzeptiert. Eine
TM, die die Wörter der Sprache akzeptiert, bei anderen Wörtern aber möglicherweise nicht hält, **erkennt** die Sprache
nur.

- Beispiel 2 erkennt die Wörter, die mit $1$ beginnen. Man kann diese Sprache aber auch entscheiden: Statt nach rechts
  zu laufen, lehnt eine andere TM bei einer $0$ einfach ab.
- Die Sprache der Programme, die anhalten, kann man erkennen (Programm ausführen und warten), aber nicht entscheiden.
  Das ist das Halteproblem.

### Links

- [Turing Machine Simulator (Ugarte)](https://turingmachinesimulator.com/)
- [Turing Machine Simulator (Morphett)](https://morphett.info/turing/turing.html)
- [JFLAP](https://www.jflap.org): Turingmaschinen zeichnen und testen

### Zusammenfassung

- Eine **Turingmaschine** hat ein unendliches Band, einen Lese-/Schreibkopf und endlich viele Zustände. Ein Übergang
  $a \mid b, R$ liest $a$, schreibt $b$ und bewegt den Kopf.
- Sie **akzeptiert**, wenn sie einen Endzustand erreicht, lehnt ab, wenn es keinen Übergang gibt, und kann auch **nie
  halten**.
- Turingmaschinen erkennen mehr als Kellerautomaten, z.B. $\{a^nb^nc^n\}$, und können Funktionen **berechnen**.
- Varianten wie mehrere Bänder oder Nichtdeterminismus können nicht mehr.
- **Church-Turing-These:** Was überhaupt berechenbar ist, ist Turing-berechenbar. Programmiersprachen wie Python sind
  **Turing-vollständig**.
- Es gibt Probleme, die keine TM lösen kann, z.B. das **Halteproblem**.